# End-to-end workflow for building the training dataset

In [2]:
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path
from scipy.ndimage import zoom
from scipy.interpolate import RegularGridInterpolator
import torch

SCENE_PATH  = Path("../../S1A_EW_GRDM_1SDH_20180124T194759_20180124T194859_020301_022AA4_1F75_icechart_dmi_201801241950_SouthEast_RIC.nc")
CHIP_SIZE   = 256 # pixels
PATCH_SIZE  = 8 # pixels per patch side
N_PATCHES   = (CHIP_SIZE // PATCH_SIZE) ** 2 # 256 patches per chip

# ── Band lists ────────────────────────────────────────────────────────────────
SAR_BANDS = [
    'nersc_sar_primary',    # HH
    'nersc_sar_secondary',  # HV
]

# Feature-contract order: ascending frequency, H before V within each pair
AMSR2_BANDS = [
    'btemp_6_9h',  'btemp_6_9v',
    'btemp_7_3h',  'btemp_7_3v',
    'btemp_10_7h', 'btemp_10_7v',
    'btemp_18_7h', 'btemp_18_7v',
    'btemp_23_8h', 'btemp_23_8v',
    'btemp_36_5h', 'btemp_36_5v',
    'btemp_89_0h', 'btemp_89_0v',
]

ERA5_BANDS = [
    'u10m_rotated', 'v10m_rotated',
    't2m', 'skt', 'tcwv', 'tclw',
]

ANCILLARY_BANDS = ['sar_grid_incidenceangle', 'distance_map']

ALL_BANDS = SAR_BANDS + AMSR2_BANDS + ERA5_BANDS

## Create assembly harness and produce the two training GeoParquet tables
### Grab the list of training scenes from S3.
Iterate through each scene, and do the following:
- Create empty chip and patch tables for the file being processed.
- Call the data loader, which reads the file and iteratively returns chips. On each chip:
    - Generate embeddings
    - Compute patch-level ancillary features
    - Process labels
    - Assemble embeddings, labels, and ancillary features into table rows. Remember there are two tables, so that’s one row per chip, and 1024 per patch.
    - Append rows to the chip / patch tables
- Once all chips are processed, write the chip and patch tables to S3 as Geoparquet files. Name the tables with the Scene ID.
- Move on to the next file


In [3]:
import boto3
import json

BUCKET = "prescient-ice-data"
S3_PREFIX = "training_data/ai4arctic/raw_train/"
STATS_KEY  = "training_data/ai4arctic/statistics/dataset_stats.json"

session = boto3.Session(profile_name="spk_data")
s3 = session.client("s3")
response = s3.get_object(Bucket=BUCKET, Key=STATS_KEY)
stats    = json.loads(response['Body'].read().decode('utf-8'))

BAND_MEANS = {var: stats[var]['mean'] for var in ALL_BANDS if var in stats}

print(f"Loaded stats for {len(BAND_MEANS)} bands from S3")

Loaded stats for 16 bands from S3


### Open the NetCDF and read everything into memory

In [4]:
# ── Open scene once ───────────────────────────────────────────────────────────
ds = xr.open_dataset(SCENE_PATH, engine='netcdf4')

print("Scene dimensions:")
print(dict(ds.sizes))
print(f"\nSAR shape: {ds['nersc_sar_primary'].shape}")
print(f"AMSR2 shape: {ds['btemp_6_9h'].shape}")
print(f"ERA5 shape: {ds['u10m_rotated'].shape}")
print(f"GCP points: {ds.sizes.get('sar_grid_points', 'not found')}")

Scene dimensions:
{'sar_lines': 10006, 'sar_samples': 10458, 'sar_grid_points': 441, '2km_grid_lines': 200, '2km_grid_samples': 209, 'polygon_codes': 27}

SAR shape: (10006, 10458)
AMSR2 shape: (200, 209)
ERA5 shape: (200, 209)
GCP points: 441


In [5]:
# ── Read SAR into memory ──────────────────────────────────────────────────────
sar_h, sar_w = ds['nersc_sar_primary'].shape

sar = {
    var: ds[var].values.astype(np.float32)
    for var in SAR_BANDS
}

print(f"SAR loaded: {sar_h} x {sar_w} pixels")

# ── Valid mask: land = distance_map code 0, nodata = NaN in SAR ───────────────
# Compute before any substitution
distance_map = ds['distance_map'].values.astype(np.float32)
is_land      = (distance_map == 0)
is_nodata    = np.isnan(sar['nersc_sar_primary'])
valid_mask   = ~is_land & ~is_nodata   # True = valid pixel

print(f"Valid pixels: {valid_mask.sum():,} / {valid_mask.size:,} "
      f"({100*valid_mask.mean():.1f}%)")
print(f"Land pixels:   {is_land.sum():,}")
print(f"Nodata pixels: {is_nodata.sum():,}")

SAR loaded: 10006 x 10458 pixels
Valid pixels: 93,892,767 / 104,642,748 (89.7%)
Land pixels:   10,005,348
Nodata pixels: 879,874


In [6]:
# ── Substitute land/nodata pixels with band mean ──────────────────────────────
# After substitution, these pixels are exactly zero in Clay's normalised space
# (because mean - mean = 0 after z-score normalisation)
for var in SAR_BANDS:
    fill = BAND_MEANS.get(var, 0.0)
    sar[var][~valid_mask] = fill

print("SAR substitution done")

SAR substitution done


### Resample ancillaries to SAR resolution

In [7]:
def resample_to_sar(arr, target_h, target_w, order=1):
    """
    Bilinear resample arr to (target_h, target_w).
    order=1 is bilinear, order=0 is nearest-neighbour.
    """
    zoom_h = target_h / arr.shape[0]
    zoom_w = target_w / arr.shape[1]
    return zoom(arr.astype(np.float32), (zoom_h, zoom_w), order=order)


# ── Resample AMSR2 ────────────────────────────────────────────────────────────
amsr2 = {}
for var in AMSR2_BANDS:
    raw        = ds[var].values.astype(np.float32)
    resampled  = resample_to_sar(raw, sar_h, sar_w)
    # Substitute invalid pixels
    fill = BAND_MEANS.get(var, 0.0)
    resampled[~valid_mask] = fill
    amsr2[var] = resampled

print(f"AMSR2 resampled: {list(amsr2.values())[0].shape}")

# ── Resample ERA5 ─────────────────────────────────────────────────────────────
era5 = {}
for var in ERA5_BANDS:
    raw       = ds[var].values.astype(np.float32)
    resampled = resample_to_sar(raw, sar_h, sar_w)
    fill = BAND_MEANS.get(var, 0.0)
    resampled[~valid_mask] = fill
    era5[var] = resampled

print(f"ERA5 resampled:  {list(era5.values())[0].shape}")

# ── Resample distance_map and incidence angle ─────────────────────────────────
# distance_map is already at SAR resolution — no resample needed
# incidence angle is on the GCP sparse grid — handled in Section 4 via interpolation

print("Ancillary resampling complete")

AMSR2 resampled: (10006, 10458)
ERA5 resampled:  (10006, 10458)
Ancillary resampling complete


### GCP interpolation for lat/lon and incidence angle

In [8]:
# ── Read GCPs ─────────────────────────────────────────────────────────────────
n_gcps = ds.dims['sar_grid_points']   # dynamic, typically 441 (21x21)
gcp_side = int(np.sqrt(n_gcps))       # 21

gcp_lines  = ds['sar_grid_line'].values        # pixel row indices
gcp_samps  = ds['sar_grid_sample'].values      # pixel col indices
gcp_lats   = ds['sar_grid_latitude'].values
gcp_lons   = ds['sar_grid_longitude'].values
gcp_angles = ds['sar_grid_incidenceangle'].values

print(f"GCPs: {n_gcps} ({gcp_side}x{gcp_side} grid)")
print(f"Line range: {gcp_lines.min():.0f} → {gcp_lines.max():.0f}")
print(f"Sample range: {gcp_samps.min():.0f} → {gcp_samps.max():.0f}")

GCPs: 441 (21x21 grid)
Line range: 0 → 10005
Sample range: 0 → 10457


/var/folders/w3/lvdn2w6d72ngld7k70v66yg40000gn/T/ipykernel_88885/1134226930.py:2: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  n_gcps = ds.dims['sar_grid_points']   # dynamic, typically 441 (21x21)


In [9]:
# ── Build 2D interpolators ────────────────────────────────────────────────────
# Sort GCPs into row-major order before reshaping — NetCDF storage order
# is not guaranteed to be row-major.
sort_idx  = np.lexsort((gcp_samps, gcp_lines))
lines_2d  = gcp_lines[sort_idx].reshape(gcp_side, gcp_side)
samps_2d  = gcp_samps[sort_idx].reshape(gcp_side, gcp_side)
lats_2d   = gcp_lats[sort_idx].reshape(gcp_side, gcp_side)
lons_2d   = gcp_lons[sort_idx].reshape(gcp_side, gcp_side)
angles_2d = gcp_angles[sort_idx].reshape(gcp_side, gcp_side)

# Row/col axes of the interpolator — use the unique line/sample values
row_axis = lines_2d[:, 0]    # one value per row
col_axis = samps_2d[0, :]    # one value per col

kw = dict(method='linear', bounds_error=False, fill_value=None)
interp_lat   = RegularGridInterpolator((row_axis, col_axis), lats_2d,   **kw)
interp_lon   = RegularGridInterpolator((row_axis, col_axis), lons_2d,   **kw)
interp_angle = RegularGridInterpolator((row_axis, col_axis), angles_2d, **kw)

def get_chip_geo(row_center, col_center):
    """Return (lat, lon, incidence_angle) for a pixel coordinate."""
    pt  = np.array([[float(row_center), float(col_center)]])
    lat = float(interp_lat(pt)[0])    # [0]: result is shape (1,), not scalar
    lon = float(interp_lon(pt)[0])
    ang = float(interp_angle(pt)[0])
    return lat, lon, ang

# Full-scene incidence angle by zooming the 21×21 GCP grid to SAR resolution
incidence_angle_full = resample_to_sar(angles_2d.astype(np.float32), sar_h, sar_w)

# Quick sanity check
lat0, lon0, ang0 = get_chip_geo(sar_h // 2, sar_w // 2)
print(f"Scene centre: lat={lat0:.3f}, lon={lon0:.3f}, incidence={ang0:.2f}°")
print(f"Incidence angle array: {incidence_angle_full.shape}, "
      f"range {incidence_angle_full.min():.1f}–{incidence_angle_full.max():.1f}°")

Scene centre: lat=64.724, lon=-35.216, incidence=34.58°
Incidence angle array: (10006, 10458), range 19.1–46.7°


In [10]:
# ── Parse SIGRID-3 CT values from polygon_codes + polygon_icechart ───────────
poly_chart  = ds['polygon_icechart'].values
poly_codes  = ds['polygon_codes'].values

header  = str(poly_codes[0]).split(';')
ct_col  = header.index('CT')

def parse_ct_tenths(ct_str):
    """Convert a raw SIGRID-3 CT string to tenths (0–10 float), or None."""
    s = ct_str.strip()
    if s == '-9':
        return None
    if '-' in s[1:]:          # range code e.g. '50-70' → midpoint 60 → 6.0
        lo, _, hi = s.partition('-')
        return (float(lo) + float(hi)) / 2.0 / 10.0
    return float(s) / 10.0

ct_lookup = {}
for row in poly_codes[1:]:
    parts = str(row).split(';')
    v = parse_ct_tenths(parts[ct_col])
    if v is not None:
        ct_lookup[int(parts[0])] = v

# polygon_icechart may be float64 (NaN at fill) or uint16 (65535 at fill)
if np.issubdtype(poly_chart.dtype, np.floating):
    valid_polygon = ~np.isnan(poly_chart)
    poly_ids_int  = np.where(valid_polygon, poly_chart.astype(np.int32), 0)
else:
    valid_polygon = (poly_chart != 65535)
    poly_ids_int  = poly_chart.astype(np.int32)

chart_ct_full = np.full((sar_h, sar_w), np.nan, dtype=np.float32)
if valid_polygon.any():
    max_pid = int(poly_ids_int[valid_polygon].max())
    ct_vec  = np.full(max_pid + 1, np.nan, dtype=np.float32)
    for pid, v in ct_lookup.items():
        if pid <= max_pid:
            ct_vec[pid] = float(v)
    chart_ct_full[valid_polygon] = ct_vec[poly_ids_int[valid_polygon]]

valid_ct = ~np.isnan(chart_ct_full)
print(f"CT coverage: {valid_ct.sum():,} / {valid_ct.size:,} pixels "
      f"({100*valid_ct.mean():.1f}%)")
print(f"CT value range: {np.nanmin(chart_ct_full):.1f} – {np.nanmax(chart_ct_full):.1f} tenths")

/var/folders/w3/lvdn2w6d72ngld7k70v66yg40000gn/T/ipykernel_88885/2109349982.py:28: RuntimeWarning: invalid value encountered in cast
  poly_ids_int  = np.where(valid_polygon, poly_chart.astype(np.int32), 0)


CT coverage: 20,034,215 / 104,642,748 pixels (19.1%)
CT value range: 1.0 – 9.2 tenths


In [11]:
# ── Clay positional encodings ─────────────────────────────────────────────────
def time_encoding(dt):
    """Clay temporal encoding: [sin/cos week, sin/cos hour] as float32."""
    week = float(dt.isocalendar().week)
    hour = dt.hour + dt.minute / 60.0
    return np.array([
        math.sin(week * 2*math.pi / 52),
        math.cos(week * 2*math.pi / 52),
        math.sin(hour * 2*math.pi / 24),
        math.cos(hour * 2*math.pi / 24),
    ], dtype=np.float32)

def latlon_encoding(lat, lon):
    """Clay spatial encoding: [sin/cos lat, sin/cos lon] as float32."""
    return np.array([
        math.sin(lat * math.pi / 180),
        math.cos(lat * math.pi / 180),
        math.sin(lon * math.pi / 180),
        math.cos(lon * math.pi / 180),
    ], dtype=np.float32)

_S1_DATETIME_RE = re.compile(r'(\d{8}T\d{6})')
m = _S1_DATETIME_RE.search(SCENE_PATH.name)
acq_dt   = datetime.strptime(m.group(1), '%Y%m%dT%H%M%S').replace(tzinfo=timezone.utc)
time_enc = time_encoding(acq_dt)
print(f"Acquisition: {acq_dt.isoformat()}")
print(f"Time encoding: {time_enc}")

# ── Chip tiling: regular grid, trailing chip shifted to align with scene edge ─
def chip_starts(scene_dim, chip_size=CHIP_SIZE):
    starts = list(range(0, scene_dim, chip_size))
    if starts[-1] + chip_size > scene_dim:
        starts[-1] = scene_dim - chip_size
    return starts

# Pre-stack for slicing efficiency
amsr2_full = np.stack([amsr2[b] for b in AMSR2_BANDS])   # (14, H, W)
era5_full  = np.stack([era5[b]  for b in ERA5_BANDS])    # (6, H, W)
dist_uint8 = distance_map.astype(np.uint8)

scene_id = SCENE_PATH.stem

def yield_chips(chip_size=CHIP_SIZE):
    row_starts = chip_starts(sar_h, chip_size)
    col_starts = chip_starts(sar_w, chip_size)
    n_total, n_skipped = len(row_starts) * len(col_starts), 0
    for row_start in row_starts:
        for col_start in col_starts:
            r0, r1 = row_start, row_start + chip_size
            c0, c1 = col_start, col_start + chip_size
            chip_valid = valid_mask[r0:r1, c0:c1]
            if not chip_valid.any():
                n_skipped += 1
                continue
            cpt = np.array([[r0 + chip_size/2.0, c0 + chip_size/2.0]])
            centroid_lat = float(interp_lat(cpt)[0])
            centroid_lon = float(interp_lon(cpt)[0])
            yield {
                'sar':             np.stack([
                                       sar['nersc_sar_primary'][r0:r1, c0:c1],
                                       sar['nersc_sar_secondary'][r0:r1, c0:c1]
                                   ]),
                'amsr2':           amsr2_full[:, r0:r1, c0:c1].copy(),
                'era5':            era5_full[:, r0:r1, c0:c1].copy(),
                'distance_map':    dist_uint8[r0:r1, c0:c1].copy(),
                'incidence_angle': incidence_angle_full[r0:r1, c0:c1].copy(),
                'valid_mask':      chip_valid.copy(),
                'chart_ct':        chart_ct_full[r0:r1, c0:c1].copy(),
                'chip_row_start':  row_start,
                'chip_col_start':  col_start,
                'time_encoding':   time_enc,
                'latlon_encoding': latlon_encoding(centroid_lat, centroid_lon),
                'centroid_lat':    centroid_lat,
                'centroid_lon':    centroid_lon,
                'scene_id':        scene_id,
                'chip_id':         f"{scene_id}_r{row_start:05d}_c{col_start:05d}",
            }
    if n_skipped:
        print(f"{scene_id}: skipped {n_skipped}/{n_total} fully-invalid chips")

print(f"Expected chips (rows): {len(chip_starts(sar_h))} × {len(chip_starts(sar_w))} cols")

Acquisition: 2018-01-24T19:47:59+00:00
Time encoding: [ 0.46472317  0.885456   -0.89297897  0.45009845]
Expected chips (rows): 40 × 41 cols


In [ ]:
# ── Validate chip iteration ───────────────────────────────────────────────────
chips = list(yield_chips())
print(f"Chips yielded: {len(chips)}")

# Shape checks
c0 = chips[0]
assert c0['sar'].shape            == (2,  CHIP_SIZE, CHIP_SIZE), c0['sar'].shape
assert c0['amsr2'].shape          == (14, CHIP_SIZE, CHIP_SIZE), c0['amsr2'].shape
assert c0['era5'].shape           == (6,  CHIP_SIZE, CHIP_SIZE), c0['era5'].shape
assert c0['distance_map'].shape   == (CHIP_SIZE, CHIP_SIZE),     c0['distance_map'].shape
assert c0['incidence_angle'].shape == (CHIP_SIZE, CHIP_SIZE),    c0['incidence_angle'].shape
assert c0['valid_mask'].shape     == (CHIP_SIZE, CHIP_SIZE),     c0['valid_mask'].shape
assert c0['chart_ct'].shape       == (CHIP_SIZE, CHIP_SIZE),     c0['chart_ct'].shape
assert c0['time_encoding'].shape  == (4,), c0['time_encoding'].shape
assert c0['latlon_encoding'].shape == (4,), c0['latlon_encoding'].shape
print("Shape checks passed")

# Edge-alignment: last chips cover the scene edge
last_row = chips[-1]['chip_row_start']
last_col = chips[-1]['chip_col_start']
assert last_row + CHIP_SIZE == sar_h, f"Row edge mismatch: {last_row}+{CHIP_SIZE} != {sar_h}"
assert last_col + CHIP_SIZE == sar_w, f"Col edge mismatch: {last_col}+{CHIP_SIZE} != {sar_w}"
print(f"Edge alignment OK: last chip at row={last_row}, col={last_col}")

# No fully-invalid chip should have been yielded
for chip in chips:
    assert chip['valid_mask'].any(), f"Fully-invalid chip in output: {chip['chip_id']}"
print("No fully-invalid chips in output")

# First chip spot-check
frac = c0['valid_mask'].mean()
print(f"First chip valid fraction: {frac:.2f}")

S1A_EW_GRDM_1SDH_20180124T194759_20180124T194859_020301_022AA4_1F75_icechart_dmi_201801241950_SouthEast_RIC: skipped 98/1640 fully-invalid chips
Chips yielded: 1542
Shape checks passed
Edge alignment OK: last chip at row=9750, col=10202
No fully-invalid chips in output
First chip valid fraction: 0.82


: 